In [0]:

CREATE OR REPLACE PROCEDURE
dbw_healthcare_de_dev.etl.usp_claim_recovery_file(
    IN p_file_id STRING,
    IN p_previous_adf_run_id STRING,
    IN p_new_adf_run_id STRING,
    IN p_expected_stage STRING,
    IN p_max_retries INT
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
    DECLARE v_rows_claimed BIGINT DEFAULT 0;

    IF p_file_id IS NULL
       OR p_previous_adf_run_id IS NULL
       OR p_new_adf_run_id IS NULL
       OR p_previous_adf_run_id = p_new_adf_run_id
       OR p_expected_stage NOT IN ('BRONZE', 'SILVER', 'GOLD')
       OR p_max_retries IS NULL
       OR p_max_retries < 1
    THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT = 'Invalid recovery claim parameters';
    END IF;

    UPDATE dbw_healthcare_de_dev.etl.file_control
    SET
        adf_run_id = p_new_adf_run_id,
        retry_count = COALESCE(retry_count, 0) + 1,
        processing_start_ts = CURRENT_TIMESTAMP(),
        last_updated_ts = CURRENT_TIMESTAMP(),
        error_message = NULL
    WHERE file_id = p_file_id
      AND adf_run_id = p_previous_adf_run_id
      AND processing_stage = p_expected_stage
      AND pipeline_status = 'IN_PROGRESS'
      AND is_enabled = TRUE
      AND COALESCE(retry_count, 0) < p_max_retries;

    GET DIAGNOSTICS v_rows_claimed = ROW_COUNT;

    SELECT
        CASE WHEN v_rows_claimed = 1
             THEN TRUE ELSE FALSE END AS claim_success,
        v_rows_claimed AS rows_claimed;
END;
